# DigiSteel - DAFEGate v4 Evaluation (YOLOv11n)
Self-contained experiment: inject DAFEGate v4 into backbone (P3) and evaluate against NEU-DET test set.

## Recorded Results (2026-07-26)

| Metric | Value |
|--------|-------|
| **mAP@0.5** | **82.0%** |
| mAP@0.5:0.95 | 46.8% |
| Precision | 72.5% |
| Recall | 79.8% |
| Training time | 1.5 hours (351 epochs, early stop at 271) |
| Total params | 2,689,827 |
| DAFEGate params | 98,817 (+3.8%) |

| Class | AP@0.5 |
|-------|--------|
| crazing | 49.1% |
| inclusion | 88.3% |
| patches | 91.7% |
| pitted_surface | 85.0% |
| rolled-in_scale | 78.8% |
| scratches | 98.9% |

## Recorded Hyperparameters

```yaml
optimizer: AdamW (lr0=0.001, lrf=0.01, momentum=0.937, weight_decay=0.0005)
epochs: 400, patience: 80, batch: 32, imgsz: 640
mosaic: 0.6, mixup: 0.05, close_mosaic: 15
degrees: 5.0, translate: 0.1, scale: 0.5, flipud: 0.5, fliplr: 0.5
hsv_h: 0.0, hsv_s: 0.4, hsv_v: 0.3
warmup_epochs: 5.0, cos_lr: True, amp: True, seed: 42
```

---

To reproduce: run cells 1→5. To evaluate saved weights only: skip cell 3, point cell 4 at `runs/best.pt`.

In [1]:
# 1. Environment & Paths
import os, time, json, torch, ultralytics
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO
import warnings
warnings.filterwarnings("ignore")

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

ROOT = Path(os.getcwd()).parent
DATA_YAML = ROOT / "configs/data/neu_det.yaml"
RUNS_DIR = ROOT / "runs/detect"
EVALS_DIR = ROOT / "evals"

print(f"PyTorch: {torch.__version__} | Ultralytics: {ultralytics.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU.")


PyTorch: 2.6.0+cu124 | Ultralytics: 8.4.95
GPU: NVIDIA RTX 2000 Ada Generation


## 2. Model Initialization & Architecture Summary
Building YOLOv11n with DAFEGate v4 at P3 (additive residual, dual-branch, channel attention).
Loading COCO pretrained weights for backbone transfer learning.


In [2]:
# 2. Inject DAFEGate v4 & Build Model
import yaml, importlib
from pathlib import Path
from ultralytics import YOLO

# Create a custom YAML with DAFEGate v4 at P3 only
yaml_content = '''

# Ultralytics YOLO, AGPL-3.0 license
# YOLOv11 object detection model with DAFEGate v4 at P3 only.

nc: 6 # number of classes (NEU-DET)
scales:
  n: [0.50, 0.25, 1024] # [depth, width, max_channels]

backbone:
  # [from, repeats, module, args]
  - [-1, 1, Conv, [64, 3, 2]] # 0-P1/2
  - [-1, 1, Conv, [128, 3, 2]] # 1-P2/4
  - [-1, 2, C3k2, [256, False, 0.25]] # 2
  - [-1, 1, Conv, [256, 3, 2]] # 3-P3/8
  - [-1, 2, C3k2, [512, False, 0.25]] # 4
  - [-1, 1, DAFEGate, []] # 5: DAFEGate v4 modulating P3 (additive residual, dual-branch)
  - [-1, 1, Conv, [512, 3, 2]] # 6-P4/16
  - [-1, 2, C3k2, [512, True]] # 7
  - [-1, 1, Conv, [1024, 3, 2]] # 8-P5/32
  - [-1, 2, C3k2, [1024, True]] # 9
  - [-1, 1, SPPF, [1024, 5]] # 10
  - [-1, 2, C2PSA, [1024]] # 11

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 12
  - [[-1, 7], 1, Concat, [1]] # 13 cat backbone P4
  - [-1, 2, C3k2, [512, False]] # 14

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]] # 15
  - [[-1, 4], 1, Concat, [1]] # 16 cat backbone P3
  - [-1, 2, C3k2, [256, False]] # 17 (P3/8-small)

  - [-1, 1, Conv, [256, 3, 2]] # 18
  - [[-1, 14], 1, Concat, [1]] # 19 cat head P4
  - [-1, 2, C3k2, [512, False]] # 20 (P4/16-medium)

  - [-1, 1, Conv, [512, 3, 2]] # 21
  - [[-1, 11], 1, Concat, [1]] # 22 cat head P5
  - [-1, 2, C3k2, [1024, True]] # 23 (P5/32-large)

  - [[17, 20, 23], 1, Detect, [nc]] # 24 Detect(P3, P4, P5)
'''

import sys
sys.path.append(str(ROOT))

# Force reload of the modified DAFEGate module
if "digisteel.modules.dafe" in sys.modules:
    del sys.modules["digisteel.modules.dafe"]

from digisteel.modules.dafe import DAFEGate
import ultralytics.nn.tasks
ultralytics.nn.tasks.DAFEGate = DAFEGate

custom_yaml_path = ROOT / "configs" / "models" / "yolov11n_dafegate.yaml"
custom_yaml_path.parent.mkdir(parents=True, exist_ok=True)
custom_yaml_path.write_text(yaml_content)

# Build the model from the custom YAML
model = YOLO(str(custom_yaml_path))

# Load pretrained COCO weights (DAFEGate layers randomly initialized)
model.load("yolo11n.pt")
print("Successfully loaded pretrained backbone. DAFEGate layers randomly initialized.")

# Parameter count
total_params = sum(p.numel() for p in model.model.parameters())
dafegate_params = sum(p.numel() for n, p in model.model.named_parameters() if any(k in n for k in ['edge_branch', 'texture_branch', 'fusion', 'channel_att', 'alpha_raw']))
print(f"Total params: {total_params:,}  (DAFEGate: {dafegate_params:,}, backbone+head: {total_params - dafegate_params:,})")

# Layer-by-layer summary
print("\n" + "=" * 60)
print("DAFEGATE V4 MODEL ARCHITECTURE")
print("=" * 60)
for i, (name, m) in enumerate(model.model.named_modules()):
    if not name or "." in name:
        continue
    children = list(m.children())
    if children:
        continue
    num_params = sum(p.numel() for p in m.parameters(recurse=False))
    tname = type(m).__name__
    print(f"  [{i:>2}] {tname:<25} {name:<20} params={num_params:>8,}")
print("=" * 60)


Transferred 78/520 items from pretrained weights
Successfully loaded pretrained backbone. DAFEGate layers randomly initialized.
Total params: 2,689,827  (DAFEGate: 98,817, backbone+head: 2,591,010)

DAFEGATE V4 MODEL ARCHITECTURE


## 3. Training Execution
Hardware-aware recipe with mosaic=0.6 (reduced from 1.0 to preserve thin linear defects like crazing).
Timestamped run name prevents overwriting previous experiments.


In [3]:
# 3. Train DAFEGate v4 Model
RUN_NAME = f"dafegate_v4_{time.strftime('%Y%m%d_%H%M')}"
train_args = {
    "data": str(DATA_YAML), "task": "detect", "epochs": 400, "patience": 80,
    "batch": 32, "imgsz": 640, "device": 0, "optimizer": "AdamW",
    "lr0": 0.001, "lrf": 0.01, "momentum": 0.937, "weight_decay": 0.0005,
    "warmup_epochs": 5.0, "mosaic": 0.6, "close_mosaic": 15, "mixup": 0.05,
    "degrees": 5.0, "translate": 0.1, "scale": 0.5, "flipud": 0.5,
    "fliplr": 0.5, "hsv_h": 0.0, "hsv_s": 0.4, "hsv_v": 0.3,
    "cos_lr": True, "deterministic": True, "amp": True, "seed": SEED,
    "workers": 4, "project": str(RUNS_DIR), "name": RUN_NAME, "exist_ok": False 
}

print(f"Starting training: {RUN_NAME}")
print("-" * 40)
t0 = time.time()
try:
    results = model.train(**train_args)
except RuntimeError as e:
    if "out of memory" in str(e).lower() or "cuda" in str(e).lower():
        print("\nCUDA Out of Memory at batch=32. Dropping to 24 and retrying...")
        import gc
        torch.cuda.empty_cache()
        gc.collect()
        train_args["batch"] = 24
        model = YOLO(str(custom_yaml_path))  # reload DAFEGate model
        model.load("yolo11n.pt")
        results = model.train(**train_args)
    else:
        raise
train_time = (time.time() - t0) / 3600
print(f"\nTraining complete in {train_time:.2f} hours.")
print(f"Best weights saved at: {RUNS_DIR / RUN_NAME / 'weights/best.pt'}")


Starting training: dafegate_v4_20260726_0941
----------------------------------------
New https://pypi.org/project/ultralytics/8.4.106 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.95  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=15, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=d:\DigiSteel-Yolo\DigiSteel-YOLO\configs\data\neu_det.yaml, degrees=5.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=400, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.0, hsv_s=0.4, hsv_v=0.3, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_

## 4. Evaluation
Load the best weights from the completed run and validate against the strictly held-out test set.


In [4]:
# 4. Evaluate on Test Set
best_pt = RUNS_DIR / RUN_NAME / "weights/best.pt"
best_model = YOLO(str(best_pt))
print("Running validation on the test split...")
val_metrics = best_model.val(data=str(DATA_YAML), split="test", imgsz=640, batch=train_args["batch"], device=0, plots=False, verbose=False)
map50 = float(val_metrics.box.map50)
map50_95 = float(val_metrics.box.map)
precision = float(val_metrics.box.mp)
recall = float(val_metrics.box.mr)
class_names = best_model.names
per_class_ap50 = {cls_name: float(val_metrics.box.ap50[cls_id]) for cls_id, cls_name in class_names.items()}
print(f"\n{'='*40}\nTEST SET METRICS\n{'='*40}")
print(f"mAP@0.5:      {map50*100:.1f}%\nmAP@0.5:0.95: {map50_95*100:.1f}%")
print(f"Precision:    {precision*100:.1f}%\nRecall:       {recall*100:.1f}%")
print("\nPer-class AP@0.5:")
for cls_name, ap in per_class_ap50.items():
    print(f"  - {cls_name:<16} {ap*100:.1f}%")


Running validation on the test split...
Ultralytics 8.4.95  Python-3.11.15 torch-2.6.0+cu124 CUDA:0 (NVIDIA RTX 2000 Ada Generation, 16380MiB)
YOLOv11n_dafegate summary (fused): 117 layers, 2,682,139 parameters, 0 gradients, 7.5 GFLOPs
WARNING val: Slow image access detected (ping: 0.10.0 ms, read: 11.420.8 MB/s, size: 18.9 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\DigiSteel-Yolo\DigiSteel-YOLO\datasets\NEU-DET\yolo\labels\test.cache... 166 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 166/166  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.7it/s 1.6s0.3s
                   all        166        371      0.725      0.798       0.82      0.468
Speed: 2.3ms preprocess, 4.9ms inference, 0.0ms loss, 0.9ms postprocess per image

TEST SET METRICS
mAP@0.5:      82.0%
mAP@0.5:0.95: 46.8%
Precision:    

## 5. Export Results
Logs the numerical outcomes to a structured JSON for later comparison.


In [5]:
# 5. Save Summary to Evals
summary = {"experiment": RUN_NAME, "training_time_hours": round(train_time, 2), "map50": map50, "map50_95": map50_95, "precision": precision, "recall": recall, "per_class_ap50": per_class_ap50, "hyperparameters": train_args}
EVALS_DIR.mkdir(exist_ok=True)
json_path = EVALS_DIR / f"{RUN_NAME}_summary.json"
json_path.write_text(json.dumps(summary, indent=2))
print(f"Metrics successfully exported to: {json_path}")


Metrics successfully exported to: d:\DigiSteel-Yolo\DigiSteel-YOLO\evals\dafegate_v4_20260726_0941_summary.json
